# Argus AML — Multi-GNN Training (GPU)

**Setup:** Runtime → Change runtime type → **T4 GPU**

**Steps:**
1. Upload your datasets (LI-Small + TransXion)
2. Install dependencies
3. Train with PNAConv + auto pos_weight + early stopping
4. Download the trained model

In [ ]:
# Step 1: Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND — change runtime to T4 GPU!'}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# Step 2: Install PyG
!pip install -q torch-geometric pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install -q scikit-learn

In [ ]:
# Step 3: Upload datasets
# Option A: Upload directly (for smaller files)
from google.colab import files
import os

os.makedirs('data', exist_ok=True)

print("Upload LI-Small_Trans.csv and tx.csv (TransXion)")
print("(This may take a few minutes for large files)")
uploaded = files.upload()

for name, content in uploaded.items():
    path = f'data/{name}'
    with open(path, 'wb') as f:
        f.write(content)
    print(f"  Saved: {path} ({len(content)/1e6:.1f} MB)")

In [ ]:
# Step 3 (Alternative): Mount Google Drive if files are there
# from google.colab import drive
# drive.mount('/content/drive')
# # Then set paths below to your Drive locations, e.g.:
# # CSV_PATHS = ['/content/drive/MyDrive/data/LI-Small_Trans.csv', '/content/drive/MyDrive/data/tx.csv']

In [ ]:
# Step 4: Set dataset paths
import glob

# Auto-detect uploaded files
all_csvs = glob.glob('data/*.csv')
print("Found CSVs:", all_csvs)

# Set paths (adjust if using Drive mount)
CSV_PATHS = all_csvs  # or manually: ['data/LI-Small_Trans.csv', 'data/tx.csv']

In [ ]:
# Step 5: The full model + training code
import argparse
import json
import logging
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import PNAConv
from torch_geometric.utils import degree
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                             recall_score, average_precision_score)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger('multignn')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')


def _progress_bar(current, total, prefix='', suffix='', width=40):
    pct = current / total
    filled = int(width * pct)
    bar = '█' * filled + '░' * (width - filled)
    print(f'\r  {prefix} |{bar}| {current}/{total} ({100*pct:.0f}%) {suffix}', end='', flush=True)
    if current >= total:
        print()


def _normalize_columns(df):
    if 'From Account' in df.columns and 'Account' not in df.columns:
        df = df.rename(columns={'From Account': 'Account', 'To Account': 'Account.1'})
    elif 'Account' in df.columns and 'Account.1' not in df.columns:
        cols = list(df.columns)
        second = [i for i, c in enumerate(cols) if c == 'Account']
        if len(second) >= 2:
            cols[second[1]] = 'Account.1'
            df.columns = cols
    return df


def _port_index(endpoint, E):
    order = np.zeros(E, dtype=np.float32)
    counter = {}
    for i, v in enumerate(endpoint):
        c = counter.get(v, 0)
        order[i] = c
        counter[v] = c + 1
    return order


def _compute_deg(edge_index, n_nodes):
    d = degree(edge_index[1], num_nodes=n_nodes, dtype=torch.long)
    max_deg = min(int(d.max().item()), 500)
    return torch.bincount(d.clamp(max=max_deg), minlength=max_deg + 1)


def _temporal_split(t_edge, train=0.75, val=0.15):
    order = torch.argsort(t_edge)
    n = len(order)
    return (order[:int(n * train)],
            order[int(n * train):int(n * (train + val))],
            order[int(n * (train + val)):])


class MultiGNN(nn.Module):
    def __init__(self, node_dim=4, edge_cont_dim=12, n_currencies=16, n_formats=8,
                 hidden=64, layers=3, dropout=0.2, emb_dim=8, deg=None):
        super().__init__()
        self.cur_emb = nn.Embedding(n_currencies + 1, emb_dim)
        self.fmt_emb = nn.Embedding(n_formats + 1, emb_dim)
        edge_in = edge_cont_dim + 2 * emb_dim
        self.edge_enc = nn.Sequential(
            nn.Linear(edge_in, hidden), nn.ReLU(), nn.Linear(hidden, hidden),
        )
        self.node_enc = nn.Linear(node_dim, hidden)

        aggregators = ['mean', 'max', 'min', 'std']
        scalers = ['identity', 'amplification', 'attenuation']
        if deg is None:
            deg = torch.ones(100, dtype=torch.long)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(layers):
            self.convs.append(PNAConv(
                in_channels=hidden, out_channels=hidden,
                aggregators=aggregators, scalers=scalers,
                deg=deg, edge_dim=hidden,
                towers=1, pre_layers=1, post_layers=1,
            ))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.drop = nn.Dropout(dropout)

        self.edge_cont_dim = edge_cont_dim
        self.head = nn.Sequential(
            nn.Linear(4 * hidden, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def _encode_edges(self, edge_attr):
        cont = edge_attr[:, :self.edge_cont_dim]
        cur = self.cur_emb(edge_attr[:, 12].long().clamp(min=0))
        fmt = self.fmt_emb(edge_attr[:, 13].long().clamp(min=0))
        return self.edge_enc(torch.cat([cont, cur, fmt], dim=-1))

    def encode_nodes(self, x, edge_index, edge_attr):
        e = self._encode_edges(edge_attr)
        h = F.relu(self.node_enc(x))
        for conv, norm in zip(self.convs, self.norms):
            h = conv(h, edge_index, e)
            h = norm(h)
            h = self.drop(F.relu(h))
        return h

    def classify_edges(self, h, label_index, edge_attr_targets):
        hs, hd = h[label_index[0]], h[label_index[1]]
        e = self._encode_edges(edge_attr_targets)
        return self.head(torch.cat([hs, hd, hs * hd, e], dim=-1)).squeeze(-1)


@torch.no_grad()
def _predict(model, x, edge_index, edge_attr, label_index, edge_attr_targets, batch_size=8192):
    model.eval()
    h = model.encode_nodes(x, edge_index, edge_attr)
    probs = []
    for s in range(0, label_index.size(1), batch_size):
        chunk = label_index[:, s:s + batch_size]
        ea = edge_attr_targets[s:s + batch_size]
        probs.append(torch.sigmoid(model.classify_edges(h, chunk, ea)).cpu())
    return torch.cat(probs).numpy()


@torch.no_grad()
def _evaluate(model, x, edge_index, edge_attr, label_index, y, batch_size=8192,
              edge_attr_targets=None, sweep=False):
    p = _predict(model, x, edge_index, edge_attr, label_index, edge_attr_targets, batch_size)
    l = y.cpu().numpy()
    auc = roc_auc_score(l, p) if len(set(l)) > 1 else float('nan')
    ap = average_precision_score(l, p) if len(set(l)) > 1 else float('nan')

    thr = 0.5
    if sweep and len(set(l)) > 1:
        best_f1, best_t = -1.0, 0.5
        for t in np.linspace(0.01, 0.95, 50):
            f1 = f1_score(l, (p >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thr = float(best_t)
    pred = (p >= thr).astype(int)
    return {
        'f1': round(float(f1_score(l, pred, zero_division=0)), 4),
        'precision': round(float(precision_score(l, pred, zero_division=0)), 4),
        'recall': round(float(recall_score(l, pred, zero_division=0)), 4),
        'auc': round(float(auc), 4) if auc == auc else None,
        'ap': round(float(ap), 4) if ap == ap else None,
        'threshold': round(thr, 3),
    }

print('Model + utilities loaded ✓')

In [ ]:
# Step 6: Build graph from datasets
MAX_ROWS = None  # None = use all data. Set to 1000000 for faster test run

dfs = []
for i, csv_path in enumerate(CSV_PATHS, 1):
    _progress_bar(i, len(CSV_PATHS), prefix='Loading', suffix=Path(csv_path).name)
    chunk = pd.read_csv(csv_path, nrows=MAX_ROWS)
    chunk = _normalize_columns(chunk)
    dfs.append(chunk)

df = pd.concat(dfs, ignore_index=True)
print(f'\nTotal rows: {len(df):,}')

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='mixed')
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

src_key = df['From Bank'].astype(str) + ':' + df['Account'].astype(str)
dst_key = df['To Bank'].astype(str) + ':' + df['Account.1'].astype(str)

keep = src_key.values != dst_key.values
df = df[keep].reset_index(drop=True)
src_key, dst_key = src_key[keep].reset_index(drop=True), dst_key[keep].reset_index(drop=True)

nodes = pd.Index(pd.unique(pd.concat([src_key, dst_key])))
node_id = {k: i for i, k in enumerate(nodes)}
src = src_key.map(node_id).to_numpy()
dst = dst_key.map(node_id).to_numpy()
N, E = len(nodes), len(df)
print(f'Graph: {N:,} accounts, {E:,} transactions')
print(f'Laundering rate: {df["Is Laundering"].mean()*100:.4f}%')

# Edge features
amount = df['Amount Paid'].to_numpy(dtype=np.float64)
log_amt = np.log1p(np.clip(amount, 0, None)).astype(np.float32)
log_amt = (log_amt - log_amt.mean()) / (log_amt.std() + 1e-6)

ts = df['Timestamp'].astype('int64').to_numpy()
t_norm = ((ts - ts.min()) / (ts.max() - ts.min() + 1)).astype(np.float32)

hour_of_day = df['Timestamp'].dt.hour.to_numpy().astype(np.float32)
hour_sin = np.sin(2 * np.pi * hour_of_day / 24).astype(np.float32)
hour_cos = np.cos(2 * np.pi * hour_of_day / 24).astype(np.float32)
day_of_week = df['Timestamp'].dt.dayofweek.to_numpy().astype(np.float32)
dow_sin = np.sin(2 * np.pi * day_of_week / 7).astype(np.float32)
dow_cos = np.cos(2 * np.pi * day_of_week / 7).astype(np.float32)

is_cross_bank = (df['From Bank'].astype(str) != df['To Bank'].astype(str)).astype(np.float32).to_numpy()

cur_codes, cur_uniq = pd.factorize(df['Receiving Currency'])
fmt_codes, fmt_uniq = pd.factorize(df['Payment Format'])

high_risk_cur = df['Receiving Currency'].isin(['Bitcoin', 'BTC', 'XRP', 'ETH']).to_numpy().astype(np.float32)
high_risk_fmt = df['Payment Format'].isin(['Cheque', 'Cash']).to_numpy().astype(np.float32)
cross_x_cur = (is_cross_bank * high_risk_cur).astype(np.float32)
cross_x_fmt = (is_cross_bank * high_risk_fmt).astype(np.float32)

out_port = _port_index(src, E)
in_port = _port_index(dst, E)
out_port_n = (out_port / (out_port.max() + 1)).astype(np.float32)
in_port_n = (in_port / (in_port.max() + 1)).astype(np.float32)

y_np = df['Is Laundering'].to_numpy(dtype=np.float32)

fwd_attr = np.stack([
    log_amt, t_norm, out_port_n, in_port_n,
    np.zeros(E, np.float32),
    hour_sin, hour_cos, dow_sin, dow_cos, is_cross_bank,
    cross_x_cur, cross_x_fmt,
    cur_codes.astype(np.float32), fmt_codes.astype(np.float32),
], axis=1)
rev_attr = fwd_attr.copy()
rev_attr[:, 2], rev_attr[:, 3] = in_port_n, out_port_n
rev_attr[:, 4] = 1.0

edge_index_np = np.concatenate([np.stack([src, dst]), np.stack([dst, src])], axis=1)
edge_attr_np = np.concatenate([fwd_attr, rev_attr], axis=0)

in_deg = np.bincount(dst, minlength=N).astype(np.float32)
out_deg = np.bincount(src, minlength=N).astype(np.float32)
recv = np.bincount(dst, weights=amount, minlength=N).astype(np.float32)
sent = np.bincount(src, weights=amount, minlength=N).astype(np.float32)
x_np = np.stack([np.log1p(in_deg), np.log1p(out_deg), np.log1p(recv), np.log1p(sent)], axis=1)
x_np = (x_np - x_np.mean(0)) / (x_np.std(0) + 1e-6)

# To GPU tensors
x = torch.tensor(x_np, dtype=torch.float).to(DEVICE)
edge_index = torch.tensor(edge_index_np, dtype=torch.long).to(DEVICE)
edge_attr = torch.tensor(edge_attr_np, dtype=torch.float).to(DEVICE)
y = torch.tensor(y_np, dtype=torch.float).to(DEVICE)
label_index = torch.tensor(np.stack([src, dst]), dtype=torch.long).to(DEVICE)
t_edge = torch.tensor(t_norm, dtype=torch.float)

deg = _compute_deg(edge_index.cpu(), N)

meta = {
    'n_nodes': int(N), 'n_edges': int(E),
    'n_currencies': int(len(cur_uniq)), 'n_formats': int(len(fmt_uniq)),
    'currencies': list(map(str, cur_uniq)), 'formats': list(map(str, fmt_uniq)),
    'node_dim': 4, 'edge_cont_dim': 12,
}

print(f'\nGraph built ✓  —  {N:,} nodes, {E:,} edges, {int(y_np.sum()):,} positives')
print(f'Edge attr shape: {edge_attr.shape}')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB' if torch.cuda.is_available() else '')

In [ ]:
# Step 7: Train!
EPOCHS = 20
HIDDEN = 64
LAYERS = 3
LR = 3e-3
BATCH_SIZE = 8192

tr, va, te = _temporal_split(t_edge, train=0.75, val=0.15)
print(f'Split — train {len(tr):,} | val {len(va):,} | test {len(te):,}')
print(f'  train positives: {int(y[tr].sum()):,} ({100*y[tr].float().mean():.4f}%)')

model = MultiGNN(meta['node_dim'], meta['edge_cont_dim'], meta['n_currencies'],
                 meta['n_formats'], hidden=HIDDEN, layers=LAYERS, deg=deg).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)

n_pos = max(float(y[tr].sum()), 1.0)
n_neg = len(tr) - n_pos
auto_pw = min(n_neg / n_pos, 200.0)
pos_w = torch.tensor([auto_pw]).to(DEVICE)
print(f'pos_weight: {auto_pw:.1f}')

Ef = label_index.size(1)
edge_attr_fwd = edge_attr[:Ef]
li_tr, y_tr, ea_tr = label_index[:, tr], y[tr], edge_attr_fwd[tr]

best_val_f1 = -1.0
best_state = None
patience_counter = 0
patience = max(EPOCHS // 3, 4)
history = []

train_start = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()
    opt.zero_grad()
    h = model.encode_nodes(x, edge_index, edge_attr)
    out = model.classify_edges(h, li_tr, ea_tr)
    loss = F.binary_cross_entropy_with_logits(out, y_tr, pos_weight=pos_w)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    opt.step()

    vm = _evaluate(model, x, edge_index, edge_attr, label_index[:, va], y[va],
                   BATCH_SIZE, edge_attr_targets=edge_attr_fwd[va])
    history.append(vm)

    star = ''
    if vm['f1'] > best_val_f1:
        best_val_f1 = vm['f1']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        star = ' ⭐'
    else:
        patience_counter += 1

    elapsed = time.time() - train_start
    eta = elapsed / epoch * (EPOCHS - epoch)
    print(f'  epoch {epoch}/{EPOCHS} — loss {loss.item():.4f} | '
          f'val F1 {vm["f1"]:.4f} AUC {vm["auc"]:.4f} P {vm["precision"]:.4f} R {vm["recall"]:.4f} '
          f'({time.time()-t0:.1f}s) ETA {eta:.0f}s{star}')

    if patience_counter >= patience and epoch >= 6:
        print(f'  Early stopping — no val F1 improvement for {patience} epochs')
        break

if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    print(f'\nRestored best model (val F1={best_val_f1:.4f})')

test_m = _evaluate(model, x, edge_index, edge_attr, label_index[:, te], y[te],
                   BATCH_SIZE, edge_attr_targets=edge_attr_fwd[te], sweep=True)

print(f'\n{"="*60}')
print(f'TEST RESULTS')
print(f'  F1:        {test_m["f1"]:.4f}')
print(f'  AUC:       {test_m["auc"]:.4f}')
print(f'  AP:        {test_m["ap"]:.4f}')
print(f'  Precision: {test_m["precision"]:.4f}')
print(f'  Recall:    {test_m["recall"]:.4f}')
print(f'  Threshold: {test_m["threshold"]:.3f}')
print(f'{"="*60}')
print(f'Total time: {time.time()-train_start:.0f}s')

In [ ]:
# Step 8: Save & download model
save_data = {
    'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
    'config': {
        'node_dim': meta['node_dim'], 'edge_cont_dim': meta['edge_cont_dim'],
        'n_currencies': meta['n_currencies'], 'n_formats': meta['n_formats'],
        'hidden': HIDDEN, 'layers': LAYERS,
    },
    'deg': deg,
    'metrics': test_m,
}
torch.save(save_data, 'multignn_model.pt')

meta_json = {'metrics': test_m, 'encoders': {
    'currencies': meta['currencies'], 'formats': meta['formats']}}
with open('multignn_meta.json', 'w') as f:
    json.dump(meta_json, f, indent=2)

print('Model saved!')
print('Downloading...')
files.download('multignn_model.pt')
files.download('multignn_meta.json')

In [ ]:
# Step 9 (Optional): Plot training history
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot([h['f1'] for h in history], 'b-o', label='F1')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('F1')
ax1.set_title('Validation F1')
ax1.grid(True, alpha=0.3)

ax2.plot([h['auc'] for h in history], 'r-o', label='AUC')
ax2.plot([h['ap'] for h in history], 'g-o', label='AP')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Score')
ax2.set_title('Validation AUC / AP')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()